In [2]:
import pandas as pd
import os

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\OGSL')

# Prepare for merging

In [ ]:
ogsl = pd.read_csv("Archambault_biodiversity_data_long_2026_03_11_CP.csv", sep=";")

column_map = {
    "datasetKey": "project_id",
    "decimalLatitudeStart": "lat_start",
    "decimalLongitudeStart": "lon_start",
    "decimalLatitudeStop": "lat_end",
    "decimalLongitudeStop": "lon_end",
    "depthA": "depth_start",
    "depthB": "depth_end",
    "eventID": "org_event_id_trawls",
    "locationID": "station_id",
    "startDate": "event_date",
    "acceptedScientificName": "valid_scientific_name",
    "phylum": "phylum",
    "class": "class",
    "family": "family",
    "aphiaID": "valid_taxon_id",
    "individualCount": "density_ind_m2",
    "biomass": "biomass_g_m2",
    "gear": "sampling_protocol"
}

output_columns = [
    "ID",
    "sourceID",
    "verbatimID",
    "sourceDatabase",
    "datasetKey",
    "datasetTitle",
    "issue",
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "decimalLatitudeStop",
    "decimalLongitudeStop",
    "footprintWKT",
    "coordinateUncertaintyInMeters",
    "hasGeospatialIssues",
    "depthA",
    "depthAType",
    "depthB",
    "depthBType",
    "gebcoDepth",
    "eventID",
    "parentEventID",
    "waterBody",
    "largeMarineEcosystem",
    "locationID",
    "startDate",
    "endDate",
    "year",
    "eventRemarks",
    "verbatimScientificName",
    "acceptedScientificName",
    "identificationQualifier",
    "lifeStage",
    "sex",
    "taxonomicStatus",
    "taxonomicIssue",
    "taxonRank",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "specificEpithet",
    "validAuthority",
    "aphiaID",
    "individualCount",
    "areaSampled",
    "areaSampledUnit",
    "samplingEffort",
    "percentCoverage",
    "biomass",
    "biomassUnit",
    "dynamicProperties",
    "basisOfRecord",
    "occurrenceRemarks",
    "gear",
    "gearCategory",
    "gearDescription",
    "gearDimensions",
    "meshSize",
    "sieveFraction",
    "sampleSizeValue",
    "sampleSizeUnit",
    "subsampleToSampleRatio",
    "subsampleSpatialLevel",
    "flagged", 
    "flaggedComment"
]

ogsl_final = pd.DataFrame(index=ogsl.index)
missing_sources = []

for col in output_columns:
    if col in column_map:
        src = column_map[col]
        if src in ogsl.columns:
            ogsl_final[col] = ogsl[src]
        else:
            ogsl_final[col] = pd.NA
            missing_sources.append(src)
    else:
        ogsl_final[col] = pd.NA

ogsl_final = ogsl_final[output_columns]
ogsl_final.to_parquet("ogsl_final.parquet", index=False)

print(f"Saved ogsl_final.parquet with {len(ogsl_final):,} rows and {len(ogsl_final.columns)} columns.")
if missing_sources:
    print("Missing source columns in ogsl:", sorted(set(missing_sources)))

Saved ogsl_final.parquet with 7,257 rows and 68 columns.


In [6]:
ogsl_final = pd.read_parquet("ogsl_final.parquet")
ogsl_final["sourceDatabase"] = "OGSL"
ogsl_final.to_parquet("ogsl_final.parquet", index=False)
print("Added sourceDatabase column with value 'OGSL' and saved ogsl_final.parquet.")

Added sourceDatabase column with value 'OGSL' and saved ogsl_final.parquet.


In [ ]:
ogsl_final = pd.read_parquet("ogsl_final.parquet")

ogsl_final.loc[ogsl_final["depthA"].notna(), "depthAType"] = "start (m)"
ogsl_final.loc[ogsl_final["depthB"].notna(), "depthBType"] = "end (m)"
ogsl_final.loc[ogsl_final["individualCount"].notna(), "areaSampled"] = "1"
ogsl_final.loc[ogsl_final["individualCount"].notna(), "areaSampledUnit"] = "m2"
ogsl_final.loc[ogsl_final["biomass"].notna(), "biomassUnit"] = "g/m2"

ogsl_final.to_parquet("ogsl_final.parquet", index=False)
ogsl_final[["depthA", "depthAType", "depthB", "depthBType", "individualCount", "areaSampled", "areaSampledUnit", "biomass", "biomassUnit"]].head()

,depthA,depthAType,depthB,depthBType,individualCount,areaSampled,biomass,biomassUnit
0,75.0,start (m),78.0,end (m),0.003600,1 m2,0.000420,g/m2
1,117.0,start (m),112.0,end (m),0.007199,1 m2,0.001742,g/m2
2,671.0,start (m),671.0,end (m),0.007679,1 m2,0.001014,g/m2
3,65.0,start (m),60.0,end (m),0.172784,1 m2,0.002315,g/m2
4,71.0,start (m),61.0,end (m),0.143000,1 m2,0.109000,g/m2


# Check data types

In [11]:
ogsl_final = pd.read_parquet("ogsl_final.parquet")

# Column type overview
column_types = pd.DataFrame({
    "column": ogsl_final.columns,
    "dtype": ogsl_final.dtypes.astype(str).values,
    "non_null_count": ogsl_final.notna().sum().values,
    "null_count": ogsl_final.isna().sum().values,
    "non_null_pct": (ogsl_final.notna().sum().values / len(ogsl_final)) * 100,
})

column_types = column_types.sort_values(
    by=["non_null_count", "column"], ascending=[False, True]
 )

column_types.to_csv("ogsl_final_column_types.csv", index=False)

column_types.head(20)

,column,dtype,non_null_count,null_count,non_null_pct
31,acceptedScientificName,str,7257,0,100.000000
45,aphiaID,int64,7257,0,100.000000
4,datasetKey,str,7257,0,100.000000
7,decimalLatitudeStart,float64,7257,0,100.000000
9,decimalLatitudeStop,float64,7257,0,100.000000
8,decimalLongitudeStart,float64,7257,0,100.000000
10,decimalLongitudeStop,float64,7257,0,100.000000
14,depthA,float64,7257,0,100.000000
15,depthAType,str,7257,0,100.000000
16,depthB,float64,7257,0,100.000000
